In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("../data/train.csv")

df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
y = df["SalePrice"]
X = df.drop("SalePrice", axis=1)


In [4]:
X = X.drop("Id", axis=1)

In [5]:
# House age
X["HouseAge"] = df["YrSold"] - df["YearBuilt"]

# Remodel age
X["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]

# Total bathrooms
X["TotalBathrooms"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"]
)

# Total square footage
X["TotalSF"] = (
    df["TotalBsmtSF"]
    + df["1stFlrSF"]
    + df["2ndFlrSF"]
)

# Total porch area
X["TotalPorchSF"] = (
    df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
    + df["WoodDeckSF"]
)

# Garage age
X["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]

# Total living area
X["TotalLivingArea"] = df["GrLivArea"] + df["TotalBsmtSF"]

In [6]:
# Remove sparse columns
X = X.drop(columns=["Alley", "PoolQC", "Fence", "MiscFeature"])

# Handle categorical missing values
cat_cols = X.select_dtypes(include="object").columns
X[cat_cols] = X[cat_cols].fillna("None")

# Handle numerical missing values
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_27728\2262734495.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [7]:
# Binary Encoding
le = LabelEncoder()
binary_cols = ["Street", "Utilities", "CentralAir"]

for col in binary_cols:
    X[col] = le.fit_transform(X[col])

# Ordinal Encoding
quality_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

ordinal_cols = [
    "ExterQual","ExterCond","BsmtQual","BsmtCond",
    "HeatingQC","KitchenQual","FireplaceQu",
    "GarageQual","GarageCond","PoolQC"
]

for col in ordinal_cols:
    if col in X.columns:
        X[col] = X[col].map(quality_map)

# One-Hot Encoding
remaining_cat_cols = X.select_dtypes(include="object").columns
X = pd.get_dummies(X, columns=remaining_cat_cols)

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_27728\3016847360.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_cat_cols = X.select_dtypes(include="object").columns


In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Selection using Random Forest
rf_selector = RandomForestRegressor(random_state=42)
rf_selector.fit(X_train, y_train)

importances = rf_selector.feature_importances_
feature_importance = pd.Series(importances, index=X_train.columns)

top_features = feature_importance.sort_values(ascending=False).head(50).index

X_train = X_train[top_features]
X_test = X_test[top_features]

In [9]:
# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Final Model (Gradient Boosting)
model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=1.0,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluation
preds = model.predict(X_test)

mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, preds)

print("Final Model Performance")
print("RMSE:", rmse)
print("R2 Score:", r2)

Final Model Performance
RMSE: 27094.403643470258
R2 Score: 0.9042926351544953


In [10]:
import joblib
joblib.dump(model, "../models/final_model.pkl")

['../models/final_model.pkl']

In [11]:
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(top_features, "../models/features.pkl")

['../models/features.pkl']

# Final Model Report

## 1. Objective

The objective of this final experiment was to build a **complete and optimized machine learning pipeline** by combining the best techniques identified in previous experiments.

This includes:

- Feature engineering
- Data preprocessing
- Encoding strategies
- Feature selection
- Model optimization

The goal is to achieve strong predictive performance while maintaining a clean and efficient pipeline.

---

# 2. Final Pipeline Overview

Raw Data  
→ Feature Engineering  
→ Remove Sparse Columns  
→ Handle Missing Values  
→ Encoding  
→ Train/Test Split  
→ Feature Selection (Top 50 Features)  
→ Feature Scaling  
→ Final Model (Gradient Boosting)  

---

# 3. Feature Engineering

The following features were created:

- HouseAge  
- RemodelAge  
- TotalBathrooms  
- TotalSF  
- TotalPorchSF  
- GarageAge  
- TotalLivingArea  

These features improved the model’s ability to capture real-world relationships between house characteristics and price.

---

# 4. Data Preprocessing

### Removing Sparse Columns

The following columns were removed due to excessive missing values:

- Alley  
- PoolQC  
- Fence  
- MiscFeature  

---

### Handling Missing Values

- **Categorical Features** → Filled with `"None"`  
- **Numerical Features** → Filled using **median**  

Median was used because it is robust to outliers.

---

# 5. Encoding Strategy

A mixed encoding approach was applied:

- **Binary Features** → Label Encoding  
- **Ordinal Features** → Mapped using quality scale  
- **Nominal Features** → One-Hot Encoding  

This approach ensures that the structure and meaning of categorical data are preserved.

---

# 6. Feature Selection

Feature selection was performed using **Random Forest feature importance**.

- Total features after encoding: **253**
- Selected features: **Top 50**

This reduced dimensionality while retaining the most important predictive information.

---

# 7. Feature Scaling

Feature scaling was applied using **StandardScaler**.

- Mean = 0  
- Standard Deviation = 1  

Scaling was applied after train-test split to avoid data leakage.

---

# 8. Final Model

The final model used was:

**Gradient Boosting Regressor**

### Best Parameters

- n_estimators = 300  
- learning_rate = 0.05  
- max_depth = 4  
- min_samples_split = 5  
- min_samples_leaf = 2  
- subsample = 1.0  

---

# 9. Final Results

| Metric | Value |
|------|------|
| RMSE | ~27094 |
| R² Score | ~0.904 |

---

# 10. Observations

- Gradient Boosting consistently outperformed all other models  
- Feature engineering significantly improved performance  
- Feature selection reduced dimensionality with minimal performance loss  
- Hyperparameter tuning provided only marginal improvement  

---

# 11. Key Insight

The most important finding from this project is:

**Feature Engineering > Hyperparameter Tuning**

This indicates that improving data representation has a greater impact than optimizing model parameters.

---

# 12. Conclusion

The final pipeline successfully combines multiple machine learning techniques to build an efficient and accurate regression model.

Key takeaways:

- Proper feature engineering is critical for performance  
- Tree-based ensemble models perform best on structured tabular data  
- Dimensionality reduction improves efficiency without significant loss of accuracy  
- Hyperparameter tuning has limited impact when the model is already well-optimized  

This pipeline serves as a strong foundation for further machine learning research and real-world applications.